In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd

## SIRS Model simplified
The model below is a simplified version of the SIRS model, useful to explain the beginning of the disease's spread - i.e. for small $t$.
The resulting equations are the following:
$$
\begin{cases}
\dot{S} = -\beta SI + \theta R \\
\dot{I} = \beta SI - \gamma I \\
\dot{R} = \gamma I - \theta R
\end{cases}
$$

We focused on the last two equations, so that the resulting model can be written as:
$$
\begin{cases}
\dot{I} = I (\beta - \gamma) -\beta RI - \beta I^{2} \\
\dot{R} = \gamma I - \theta R
\end{cases}
$$

In [ ]:
def SIRS_small_t(t, X, beta, gamma, theta):
    I, R = X

    dI = I*(beta - gamma) - beta * R * I - beta * I**2
    dR = gamma * I - theta * R

    return [dI, dR]

In [ ]:
t_span = (0, 120)
t = np.linspace(t_span[0], t_span[1], 10000)

In [ ]:
# t from the study
# R0 < 1
gamma = 0.5
mu = 0.
theta = 2 / 12
beta = 4.7*(mu + gamma)

S0 = 0.99
I0 = 0.01
R0 = 0

X0 = [I0, R0]

solution_min = solve_ivp(SIRS_small_t, t_span, X0, args=(beta, gamma, theta), dense_output=True)
sol_min = solution_min.sol(t)

#fig_min.show()

In [ ]:
# t from the study
# R0 > 1
gamma = 0.5
mu = 0.
theta = 0.05 / 12
beta = 2.2*(mu + gamma)

S0 = 0.99
I0 = 0.01
R0 = 0

X0 = [I0, R0]

solution_max = solve_ivp(SIRS_small_t, t_span, X0, args=(beta, gamma, theta), dense_output=True)
sol_max = solution_max.sol(t)

#fig_max.show()

In [ ]:
fig_min = px.line(x=t, y=[0] * len(sol_min[0]), title=f'SIRS simplified model - R0={0.7}',
                  labels={'x': 'Time (Months)', 'y': 'Fraction of population'},
                  color_discrete_sequence=['rgba(0, 0, 0, 0)'])
fig_min.add_scatter(x=t, y=1 - sol_min[1] - sol_min[0], mode='lines', name='Susceptibles',
                    line=dict(color='rgba(0, 0, 255, 0.3)'))
fig_min.add_scatter(x=t, y=sol_min[0], mode='lines', name='Infected', line=dict(color='red'))
fig_min.add_scatter(x=t, y=sol_min[1], mode='lines', name='Recovered', line=dict(color='rgba(0, 255, 0, 0.6)'))

fig_min.update_layout(template='plotly_white')

fig_max = px.line(x=t, y=[0] * len(sol_max[0]), title=f'SIRS simplified model - R0={round(beta / (mu + gamma), 3)}',
                  labels={'x': 'Time (Months)', 'y': 'Fraction of population'},
                  color_discrete_sequence=['rgba(0, 0, 0, 0)'])
fig_max.add_scatter(x=t, y=1 - sol_max[1] - sol_max[0], mode='lines', name='Susceptibles',
                    line=dict(color='rgba(0, 0, 255, 0.3)'))
fig_max.add_scatter(x=t, y=sol_max[0], mode='lines', name='Infected', line=dict(color='red'))
fig_max.add_scatter(x=t, y=sol_max[1], mode='lines', name='Recovered', line=dict(color='rgba(0, 255, 0, 0.6)'))

fig_max.update_layout(template='plotly_white')



fig = make_subplots(rows = 1, cols = 2, shared_yaxes=True, subplot_titles=(f"SIRS simplified model - R0=0.1", f"SIRS simplified model - R0={round(beta / (mu + gamma), 3)}") )
fig.update_layout(width=1000, height=600, template='plotly_white')

for trace in fig_min.data:
    fig.add_trace(trace, row=1, col=1)

for trace in fig_max.data:
    fig.add_trace(trace, row=1, col=2)

legend_seen = {}
for trace in fig.data:
    if trace.name in legend_seen:
        trace.showlegend = False
    else:
        legend_seen[trace.name] = True

fig.show()


In [ ]:
# Time points for which to get the solution
t = np.linspace(t_span[0], t_span[1], 100000)
solution = solve_ivp(SIRS_small_t, t_span, X0, args=(beta, gamma, theta), dense_output=True)
sol = solution.sol(t)
fig_model = px.line(x=t, y=[0] * len(sol[0]), title=f'SIRS simplified model - R0={round(beta / (mu + gamma), 3)}',
                    labels={'x': 'Time (Months)', 'y': 'Fraction of population'},
                    color_discrete_sequence=['rgba(0, 0, 0, 0)'], width=600, height=600)
# fig_model.add_scatter(x=t, y=1 - sol[1] - sol[0], mode='lines', name='Susceptibles',
#                       line=dict(color='rgba(0, 0, 255, 0.3)'))
fig_model.add_scatter(x=t, y=sol[0], mode='lines', name='Infected', line=dict(color='red'))
# fig_model.add_scatter(x=t, y=sol[1], mode='lines', name='Recovered', line=dict(color='rgba(0, 255, 0, 0.6)'))

fig_model.update_layout(template='plotly_white')

fig_model.show()

In [ ]:
fig = px.scatter(x=t, y=np.exp( (beta - gamma)*t), title=f'SIRS Analytical Behaviour', width=600, height=600, size=2)

## SIRS Model

The model studied below is the following:
$$
\begin{cases}
\dot{S} = (\mu + \theta) - (\mu + \theta)S - \beta SI - \theta I \\
\dot{I} = I(\beta S - (\mu + \gamma)) \\
\dot{R} = \gamma I - (\mu + \theta)R
\end{cases}
$$

In particular we focused on the last two equations, so that the resulting model can be written as:
$$
\begin{cases}
\dot{I} = I(\beta S - (\mu + \gamma)) - \beta RI - \beta I^{2} \\
\dot{R} = \gamma I - (\mu + \theta)R
\end{cases}
$$

In [ ]:
def SIRS_IR (t, X, beta, gamma, mu, theta):
    I, R = X

    dI = I*(beta - (mu + gamma)) - beta * R * I - beta * I**2
    dR = gamma * I - (mu + theta) * R

    return [dI, dR]

In [ ]:
# Parameters
# These parameters are taken from Grassly 2005 and are related to syphilis cases in the USA

# t from the study
gamma = 6/12
mu = 0.03 / 12
theta = 0.05 / 12
beta = 2.5*(mu + gamma)

# Initial conditions
# S0 = 0.999                      # Initial suspicious population
# I0 = 0.001                      # Initial infected population
# R0 = 0                          # Initial recovered population

S0 = 0.99
I0 = 0.01
R0 = 0


X0 = [I0, R0]

In [ ]:
# Some specifics of the model
R_0 = round(beta / (mu+gamma), 8)

damping = -(theta + mu)*(theta + beta)/2/(mu + gamma + theta)

period_body = (mu + theta)*(mu + gamma)*(R_0 - 1) - ( (mu + theta)*(theta + beta) / (2*(mu + theta + gamma)) )**2
period = 2*np.pi / np.sqrt(period_body) if period_body > 0 else 0

# this period function is from Keeling, Rohani 20080
# period_si = 4*np.pi / np.sqrt( 4 * (R_0-1) * (mu + gamma) * (mu + theta) - (mu + theta + (mu + theta)*(beta - mu - gamma)/(mu + gamma + theta))**2 )

# Endemic equilibrium
S_e = (mu + gamma) / beta
I_e = (1 - 1/R_0) * (mu + theta) / (mu + gamma + theta)
R_e = (1 - 1/R_0) * gamma / (mu + gamma + theta)

In [ ]:
# print parameters
print(f"Parameters: \nBeta: {round(beta, 5)}\t\t Gamma: {gamma}\t\t Mu: {mu}\nTheta: {round(theta, 5)}\t\tR0: {round(R_0, 3)}\n")
print(f"Specifics: \nDamping: {round(damping, 5)}\t Oscillation period: {round(period, 5)}\nEndemic Equilibrium: {(round(S_e, 5), round(I_e, 5), round(R_e, 5))}")
# print(f"Oscillation SI period: {period_si}")

In [ ]:
# Time span
# t = 1 day
# t_span = (0, 50000)

# t = 1 month
t_span = (0, 1200)

# Solve the system of ODEs
solution = solve_ivp(SIRS_IR, t_span, X0, args=(beta, gamma, mu, theta), dense_output=True)

# Time points for which to get the solution
t = np.linspace(t_span[0], t_span[1], 50000)
sol = solution.sol(t)

In [ ]:
fig_model = px.line(x=t, y=[0]*len(sol[0]), title='SIRS Model', labels={'x': 'Time', 'y': 'Population'}, color_discrete_sequence=['rgba(0, 0, 0, 0)'])
fig_model.add_scatter(x=t, y=1 - sol[1] - sol[0], mode='lines', name='Susceptibles', line=dict(color='blue'))
fig_model.add_scatter(x=t, y=sol[0], mode='lines', name='Infected', line=dict(color='red'))
fig_model.add_scatter(x=t, y=sol[1], mode='lines', name='Recovered', line=dict(color='green'))
fig_model.show()

In [ ]:
beta_2 = 0.5*(mu + gamma)

solution_2 = solve_ivp(SIRS_IR, t_span, X0, args=(beta_2, gamma, mu, theta), dense_output=True)
sol_2 = solution_2.sol(t)

fig_2 = px.line(x=t, y=[0] * len(sol[0]), title=f'SIRS Dynamics, R₀ = {0.5}',
                  labels={'x': 'Time (Months)', 'y': 'Fraction of population'},
                  color_discrete_sequence=['rgba(0, 0, 0, 0)'])
fig_2.add_scatter(x=t, y=1 - sol_2[1] - sol_2[0], mode='lines', name='Susceptibles',
                    line=dict(color='rgba(0, 0, 255, 0.6)'))
fig_2.add_scatter(x=t, y=sol_2[0], mode='lines', name='Infected', line=dict(color='red'))
fig_2.add_scatter(x=t, y=sol_2[1], mode='lines', name='Recovered', line=dict(color='rgba(0, 125, 0, 1)'))

fig_2.update_layout(template='plotly_white')

fig = px.line(x=t, y=[0] * len(sol[0]), title=f'SIRS Dynamics, R₀ = {round(beta / (mu + gamma), 2)}',
                  labels={'x': 'Time (Months)', 'y': 'Fraction of population'},
                  color_discrete_sequence=['rgba(0, 0, 0, 0)'])
fig.add_scatter(x=t, y=1 - sol[1] - sol[0], mode='lines', name='Susceptibles',
                    line=dict(color='rgba(0, 0, 255, 0.6)'))
fig.add_scatter(x=t, y=sol[0], mode='lines', name='Infected', line=dict(color='red'))
fig.add_scatter(x=t, y=sol[1], mode='lines', name='Recovered', line=dict(color='rgba(0, 125, 0, 1)'))

fig.update_layout(template='plotly_white')



fig_tot = make_subplots(rows = 1, cols = 2, shared_yaxes=True, subplot_titles=(f"SIRS Dynamics,  R₀ = 0.5", f"SIRS Dynamics, R₀ = {round(beta / (mu + gamma), 2)}") )
fig_tot.update_layout(width=1000, height=600, template='plotly_white')

for trace in fig_2.data:
    fig_tot.add_trace(trace, row=1, col=1)

for trace in fig.data:
    fig_tot.add_trace(trace, row=1, col=2)

legend_seen = {}
for trace in fig_tot.data:
    if trace.name in legend_seen:
        trace.showlegend = False
    else:
        legend_seen[trace.name] = True

fig_tot.show()



In [ ]:
fig_model = px.line(x=t, y=[0]*len(sol[0]), title='SIRS - IR Model', labels={'x': 'Time', 'y': 'Population'}, color_discrete_sequence=['rgba(0, 0, 0, 0)'])
fig_model.add_scatter(x=t, y=1 - sol[1] - sol[0], mode='lines', name='Susceptibles', line=dict(color='blue'))
fig_model.add_scatter(x=t, y=sol[1], mode='lines', name='Recovered', line=dict(color='green'))
fig_model.show()

In [ ]:
fig_model_s = px.line(x=t, y=1 - sol[1] - sol[0], title='SIRS Model - Susceptibles', labels={'x': 'Time', 'y': 'Susceptibles'}, color_discrete_sequence=['blue'])
fig_model_s.add_scatter( x=t, y=[S_e]*len(sol[0]), name='Susceptibles EE', mode='lines', line=dict(color='rgba(0, 0, 0, 0.5)') )
fig_model_s.show()

In [ ]:
fig_model_i = px.line(x=t, y=sol[0], title='SIRS Model - Infectious', labels={'x': 'Time', 'y': 'Infectious'}, color_discrete_sequence=['red'])
fig_model_i.add_scatter( x=t, y=[I_e]*len(sol[0]), name='Infectious EE', mode='lines', line=dict(color='rgba(0, 0, 0, 0.2)') )
fig_model_i.show()

In [ ]:
fig_model_r = px.line(x=t, y=sol[1], title='SIRS Model - Recovered', labels={'x': 'Time', 'y': 'Recovered'}, color_discrete_sequence=['green'])
fig_model_r.add_scatter( x=t, y=[R_e]*len(sol[0]), name='Recovered EE', mode='lines', line=dict(color='rgba(0, 0, 0, 0.2)') )
fig_model_r.show()

In [ ]:
file = pd.read_csv('syphilis_sample.csv', header=None, names=['Year', 'Cases'])
#file.head(5)

In [ ]:
years = file['Year']
cases = file['Cases']
cases = [cases[i] / 5000 for i in range(len(cases))]

In [ ]:
fig = px.line(x=years, y=cases, width=800, height = 600, title='Syphilis normalized cases in the USA (1941-2001)', labels={'x':'Year','y':'Normalized Population'})
fig.show()

In [ ]:
fig_i = px.line(x=t[23500:57000], y=sol[0][23500:57000], width=800, height = 600, title='Syphilis Model ', labels={'x':'Months','y':'Normalized Population'})
# fig_i.add_scatter(x=years, y=cases, mode='lines')
fig_i.show()

In [ ]:
# phase plot
Is = [sol[0][i]/I_e for i in range(len(sol[0]))]
Rs = [sol[1][i]/R_e for i in range(len(sol[1]))]
fig_phase_1 = px.scatter(x=Is, y=Rs, title='Phase plot', labels={'x': 'I(t) / Ie', 'y': 'R(t) / Re'}, color=t)

fig_phase_1.update_layout(
    xaxis = dict(
        tickmode = 'linear',
        dtick = 1
    )
)

fig_phase_1.show()

In [ ]:
# phase plot
Ss = [(1-sol[0][i]-sol[1][i])/S_e for i in range(len(sol[0]))]

fig_phase_2 = px.scatter(x=Is, y=Ss, title='Phase plot', labels={'x': 'I(t) / Ie', 'y': 'S(t) / Se'}, color=t)
fig_phase_2.show()